# XAUUSD — Pipeline completo: Fourier → Dirección → Monte Carlo → Cadenas de Markov

**Orden del análisis:**
1. **Transformada de Fourier** sobre los datos H1 reales:
   - espectro de potencia (Welch) → **ciclos dominantes** del precio,
   - **periodicidad intradía** (perfil de retorno y volatilidad por hora del día),
   - **filtrado de ruido** (pasa-bajos en frecuencia: solo los ciclos significativos),
   - **detección de cambios de régimen** vía espectro rodante (entropía espectral + peso de baja frecuencia).
2. **Veredicto de dirección**: la extrapolación de los ciclos filtrados dice si el sesgo es **ARRIBA / ABAJO / LATERAL**.
3. **Monte Carlo** (10,000 caminos): la distribución del precio en el horizonte.
4. **Cadenas de Markov (HMM de 3 regímenes)**: la ruta hora a hora más probable con sus bandas.

**Cómo usarlo:** menú `Entorno de ejecución → Ejecutar todas`. Ajusta `CONFIG` y `TARGET` en la celda 2.

> ⚠️ **Lee esto sobre los pronósticos:** el resultado es una *distribución de probabilidad*, no una promesa.
> La banda del 80% falla por diseño 1 de cada 5 veces, y ningún modelo estadístico anticipa noticias
> (Fed, datos de empleo, geopolítica). Úsalo como mapa de escenarios y gestión de riesgo, no como certeza.


In [ ]:
# @title 1) Instalar dependencias
%pip install -q -U hmmlearn yfinance curl_cffi scipy
print("Dependencias listas.")


In [ ]:
# @title 2) Configuración
CONFIG = {
    "n_states":         3,       # regímenes del HMM
    "n_paths":          10000,   # simulaciones Monte Carlo
    "horizon_hours":    48,      # horizonte del pronóstico (velas H1)
    "checkpoint_every": 4,       # puntos de control de la ruta (cada 4 horas)
    "confidence":       0.80,    # banda de confianza para la ruta
    "ou_kappa":         0.03,    # reversión a la media en régimen rango
    "seed":             42,
    # ---- parámetros de Fourier
    "fft_fit_bars":     2048,    # cuántas velas usar para ajustar los ciclos
    "fft_min_period":   8.0,     # ignorar ciclos más cortos que esto (horas) = filtrar ruido
    "fft_n_cycles":     6,       # cuántos ciclos dominantes conservar
    "spec_window":      240,     # ventana del espectro rodante (horas)
    "spec_step":        24,      # paso del espectro rodante (horas)
    "use_cycle_drift":  True,    # sumar el ciclo extrapolado como drift en el Monte Carlo
}

# Precio objetivo opcional (ej. 4280.0) o None
TARGET = None

DATA_SOURCE = "yahoo"   # "yahoo" o "csv"
YAHOO_PERIOD = "1y"     # "3mo", "6mo", "1y", "2y" (máx ~730 días en H1)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal as sps
print("Configuración cargada.")


In [ ]:
# @title 3) Datos H1 de XAUUSD (Yahoo Finance con reintentos, o CSV de MT5)
import time

MIN_BARS = 1000   # mínimo de velas para que el análisis espectral tenga sentido


def _yahoo_api_directa(ticker, interval="1h", range_="1y"):
    """Respaldo: API de gráficos de Yahoo, con marca de tiempo."""
    import requests
    url = f"https://query1.finance.yahoo.com/v8/finance/chart/{ticker}"
    r = requests.get(url, params={"interval": interval, "range": range_},
                     headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"},
                     timeout=30)
    r.raise_for_status()
    res = r.json()["chart"]["result"][0]
    ts = pd.to_datetime(res["timestamp"], unit="s", utc=True)
    cl = res["indicators"]["quote"][0]["close"]
    return pd.Series(cl, index=ts, dtype="float64").dropna()


def load_yahoo(period="1y"):
    import yfinance as yf
    tickers = ["XAUUSD=X", "GC=F"]
    periods = [period, "6mo", "3mo"]
    last_err = None
    for attempt in range(1, 4):
        for ticker in tickers:
            for per in periods:
                try:
                    df = yf.download(ticker, interval="1h", period=per,
                                     progress=False, auto_adjust=True)
                    if df is not None and len(df) > MIN_BARS:
                        s = pd.to_numeric(df["Close"].squeeze(), errors="coerce").dropna()
                        print(f"[DATOS] {len(s)} velas H1 de {ticker} (yfinance, {per})")
                        return s
                except Exception as e:
                    last_err = e
                    print(f"[DATOS] yfinance falló con {ticker} {per}: {e}")
                try:
                    s = _yahoo_api_directa(ticker, "1h", per)
                    if len(s) > MIN_BARS:
                        print(f"[DATOS] {len(s)} velas H1 de {ticker} (API directa, {per})")
                        return s
                except Exception as e:
                    last_err = e
                    print(f"[DATOS] API directa falló con {ticker} {per}: {e}")
        if attempt < 3:
            wait = 30 * attempt
            print(f"[DATOS] Yahoo está limitando peticiones; espero {wait}s y reintento...")
            time.sleep(wait)
    raise RuntimeError(
        f"No pude descargar datos. Último error: {last_err}\n"
        "Opciones: espera unos minutos; o 'Entorno de ejecución -> Desconectar y "
        "eliminar entorno' (cambia la IP); o DATA_SOURCE='csv'.")


def load_mt5_csv_upload():
    from google.colab import files
    print("Sube tu CSV exportado de MT5 (XAUUSD H1)...")
    uploaded = files.upload()
    path = list(uploaded.keys())[0]
    for sep in ["\t", ",", ";"]:
        try:
            df = pd.read_csv(path, sep=sep)
            if df.shape[1] >= 4:
                break
        except Exception:
            continue
    else:
        raise ValueError("No pude leer el CSV. Verifica el separador.")
    df.columns = [c.strip().strip("<>").upper() for c in df.columns]
    if "CLOSE" not in df.columns:
        raise ValueError(f"No encuentro la columna CLOSE. Columnas: {list(df.columns)}")
    closes = pd.to_numeric(df["CLOSE"], errors="coerce")
    # intentar construir marca de tiempo (para el análisis intradía)
    idx = None
    if "DATE" in df.columns and "TIME" in df.columns:
        idx = pd.to_datetime(df["DATE"].astype(str) + " " + df["TIME"].astype(str),
                             errors="coerce")
    elif "DATE" in df.columns:
        idx = pd.to_datetime(df["DATE"].astype(str), errors="coerce")
    if idx is not None and idx.notna().mean() > 0.9:
        s = pd.Series(closes.values, index=idx).dropna()
    else:
        s = closes.dropna()
        print("[DATOS] CSV sin fecha/hora utilizable: se omitirá el perfil intradía")
    print(f"[DATOS] {len(s)} velas cargadas de {path}")
    return s


series = load_yahoo(YAHOO_PERIOD) if DATA_SOURCE == "yahoo" else load_mt5_csv_upload()
closes = series.values.astype(float)
returns = np.diff(np.log(closes))
has_time = isinstance(series.index, pd.DatetimeIndex)
print(f"[DATOS] Último cierre: {closes[-1]:.2f}")
if has_time:
    print(f"[DATOS] Rango: {series.index[0]} → {series.index[-1]}")
print("[NOTA] Las velas se tratan como consecutivas (los huecos de fin de semana "
      "se ignoran, práctica estándar en análisis espectral de mercados).")


In [ ]:
# @title 4) FOURIER (a): espectro de potencia — ciclos dominantes del precio y de la volatilidad
def espectro_welch(x, nper_max=1024):
    x = np.asarray(x, dtype=float)
    x = x - x.mean()
    nper = int(min(nper_max, 2 * (len(x) // 4)))
    freqs, psd = sps.welch(x, fs=1.0, nperseg=nper)
    return freqs[1:], psd[1:]   # sin la frecuencia cero


def ciclos_dominantes(freqs, psd, n=8, min_period=3.0):
    peaks, _ = sps.find_peaks(psd)
    periodos = 1.0 / freqs[peaks]
    ok = periodos >= min_period
    peaks, periodos = peaks[ok], periodos[ok]
    orden = np.argsort(psd[peaks])[::-1][:n]
    total = psd.sum()
    return [(periodos[i], 100.0 * psd[peaks[i]] / total) for i in orden]


# --- espectro de los RETORNOS (ciclos de dirección del precio)
f_r, p_r = espectro_welch(returns)
cic_ret = ciclos_dominantes(f_r, p_r, n=8)

# --- espectro del VALOR ABSOLUTO de los retornos (ciclos de volatilidad)
f_v, p_v = espectro_welch(np.abs(returns))
cic_vol = ciclos_dominantes(f_v, p_v, n=5)

print("===== CICLOS DOMINANTES (espectro de Welch) =====")
print("  En el PRECIO (retornos):")
for per, pot in cic_ret:
    dias = per / 24.0
    print(f"    ciclo de {per:7.1f} h (~{dias:5.2f} días)  | {pot:5.2f}% de la potencia")
print("  En la VOLATILIDAD (|retornos|):")
for per, pot in cic_vol:
    print(f"    ciclo de {per:7.1f} h (~{per/24.0:5.2f} días)  | {pot:5.2f}% de la potencia")
print("\n  Lectura: en oro es típico que la VOLATILIDAD tenga un ciclo fuerte de ~24h")
print("  (sesiones Asia/Europa/NY). Ciclos de PRECIO estables son más raros: si ningún")
print("  ciclo concentra mucha potencia, el mercado es mayormente ruido + tendencia.")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, (f, p, tit) in zip(axes, [(f_r, p_r, "Espectro de los retornos (precio)"),
                                  (f_v, p_v, "Espectro de |retornos| (volatilidad)")]):
    ax.semilogx(1.0 / f, p, lw=1.2, color="#1565c0")
    for h in [24, 12, 8, 6]:
        ax.axvline(h, color="#c62828", ls=":", lw=0.8)
        ax.text(h, ax.get_ylim()[1] * 0.92, f"{h}h", color="#c62828",
                fontsize=8, ha="center")
    ax.set_xlabel("período del ciclo (horas, escala log)")
    ax.set_ylabel("densidad de potencia")
    ax.set_title(tit)
fig.tight_layout(); plt.show()


In [ ]:
# @title 5) FOURIER (b): periodicidad intradía — qué horas mueven a XAUUSD
if has_time:
    idx_ret = series.index[1:]
    horas = idx_ret.hour
    df_h = pd.DataFrame({"hora": horas, "ret": returns})
    perfil = df_h.groupby("hora")["ret"].agg(ret_medio="mean", vol="std", n="count")

    top_vol = perfil["vol"].sort_values(ascending=False).head(4)
    print("===== PERIODICIDAD INTRADÍA (hora UTC) =====")
    print("  Horas MÁS volátiles (donde se mueve el precio):")
    for h, v in top_vol.items():
        print(f"    {h:02d}:00 UTC | vol {v*100:.3f}%/h")
    print("  Horas menos volátiles:",
          ", ".join(f"{h:02d}:00" for h in perfil["vol"].sort_values().head(3).index))
    print("  (Típico: máxima actividad en el solape Londres-NY ~13-17 UTC,")
    print("   mínima en el cierre de NY / apertura de Asia ~21-01 UTC)")

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].bar(perfil.index, perfil["vol"] * 100, color="#1565c0", alpha=0.85)
    axes[0].set_title("Volatilidad por hora del día (UTC)")
    axes[0].set_xlabel("hora UTC"); axes[0].set_ylabel("desv. típica del retorno (%)")
    colores = ["#2e7d32" if v >= 0 else "#c62828" for v in perfil["ret_medio"]]
    axes[1].bar(perfil.index, perfil["ret_medio"] * 100, color=colores, alpha=0.85)
    axes[1].axhline(0, color="k", lw=0.8)
    axes[1].set_title("Retorno medio por hora del día (UTC)")
    axes[1].set_xlabel("hora UTC"); axes[1].set_ylabel("retorno medio (%)")
    fig.tight_layout(); plt.show()
else:
    print("[INTRADÍA] Sin marca de tiempo en los datos: sección omitida.")


In [ ]:
# @title 6) FOURIER (c): filtrado de ruido y extrapolación → dirección ARRIBA / ABAJO / LATERAL
N_FIT  = int(min(CONFIG["fft_fit_bars"], len(closes)))
H      = CONFIG["horizon_hours"]
x      = np.log(closes[-N_FIT:])
t      = np.arange(N_FIT)

# 1) tendencia lineal + 2) ciclos dominantes sobre el residuo (el resto es ruido)
coef    = np.polyfit(t, x, 1)
detrend = x - np.polyval(coef, t)
F   = np.fft.rfft(detrend)
fr  = np.fft.rfftfreq(N_FIT, d=1.0)
amp = np.abs(F) / N_FIT

valido = (fr > 0) & (1.0 / np.maximum(fr, 1e-12) >= CONFIG["fft_min_period"]) \
                  & (1.0 / np.maximum(fr, 1e-12) <= N_FIT / 2)
orden  = [i for i in np.argsort(amp)[::-1] if valido[i]][:CONFIG["fft_n_cycles"]]

print("===== CICLOS CONSERVADOS (el resto se descarta como ruido) =====")
for i in orden:
    print(f"  período {1.0/fr[i]:7.1f} h (~{1.0/fr[i]/24:5.2f} días) | "
          f"amplitud ±{2*amp[i]*100:.3f}% del precio")

# reconstrucción suavizada (pasado) y extrapolación (futuro)
t_ext = np.arange(N_FIT + H)
recon = np.polyval(coef, t_ext)
for i in orden:
    A, ph = 2.0 * amp[i], np.angle(F[i])
    recon += A * np.cos(2 * np.pi * fr[i] * t_ext + ph)
suave, proyeccion = recon[:N_FIT], recon[N_FIT:]

# veredicto de dirección: cambio proyectado vs volatilidad típica del horizonte
delta   = proyeccion[-1] - recon[N_FIT - 1]
sigma_h = np.std(returns[-500:]) * np.sqrt(H)
ratio   = delta / sigma_h
if   ratio >  0.5: veredicto = "ARRIBA"
elif ratio < -0.5: veredicto = "ABAJO"
else:              veredicto = "LATERAL"

precio_proy = float(np.exp(proyeccion[-1]))
print(f"\n===== VEREDICTO FOURIER A {H}h =====")
print(f"  Cambio proyectado por tendencia+ciclos: {delta*100:+.2f}% "
      f"(≈ {precio_proy:.2f} USD)")
print(f"  Volatilidad típica del horizonte:       {sigma_h*100:.2f}%")
print(f"  Señal / ruido: {ratio:+.2f}  →  DIRECCIÓN: >>> {veredicto} <<<")
print("  (|señal/ruido| < 0.5 se considera LATERAL: los ciclos no dominan al azar)")

# drift determinista del ciclo para inyectarlo al Monte Carlo (opcional)
cycle_drift = np.diff(recon[N_FIT - 1:]) if CONFIG["use_cycle_drift"] else None

VER = 400  # velas a mostrar
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(t[-VER:], np.exp(x[-VER:]), lw=0.8, color="#9e9e9e", label="precio real (con ruido)")
ax.plot(t[-VER:], np.exp(suave[-VER:]), lw=2, color="#1565c0",
        label="tendencia + ciclos (ruido filtrado)")
ax.plot(np.arange(N_FIT, N_FIT + H), np.exp(proyeccion), lw=2, ls="--", color="#c62828",
        label=f"extrapolación de ciclos ({H}h)")
ax.axvline(N_FIT - 1, color="k", ls=":", lw=1)
ax.set_title(f"Filtrado de ruido por Fourier y proyección → {veredicto}")
ax.set_xlabel("velas H1"); ax.set_ylabel("USD/oz"); ax.legend(loc="best")
fig.tight_layout(); plt.show()


In [ ]:
# @title 7) FOURIER (d): cambios de régimen — espectro rodante + HMM de Markov
# --- (i) régimen espectral: entropía y peso de baja frecuencia en ventana móvil
win, step = CONFIG["spec_window"], CONFIG["spec_step"]
r_c = returns - returns.mean()
ent, low, centros = [], [], []
for ini in range(0, len(r_c) - win + 1, step):
    seg = r_c[ini:ini + win]
    f2, p2 = sps.welch(seg, fs=1.0, nperseg=min(128, win))
    p = p2[1:] / max(p2[1:].sum(), 1e-18)
    ent.append(float(-(p * np.log(p + 1e-12)).sum() / np.log(len(p))))
    low.append(float(p2[(f2 > 0) & (f2 <= 1.0 / 24)].sum() / max(p2[1:].sum(), 1e-18)))
    centros.append(ini + win)
ent, low = np.array(ent), np.array(low)

# cambios de régimen = saltos grandes de la entropía espectral
d_ent = np.abs(np.diff(ent))
umbral = np.median(d_ent) + 2.5 * d_ent.std()
cambios = [centros[i + 1] for i in np.where(d_ent > umbral)[0]]

print("===== RÉGIMEN ESPECTRAL (ventana de", win, "h) =====")
print(f"  Entropía espectral actual: {ent[-1]:.3f} "
      f"(mediana histórica {np.median(ent):.3f})")
print(f"  Peso de ciclos lentos (>24h) actual: {low[-1]*100:.1f}% "
      f"(mediana {np.median(low)*100:.1f}%)")
if ent[-1] < np.percentile(ent, 30):
    print("  → espectro ORDENADO: domina estructura (tendencia/ciclo)")
elif ent[-1] > np.percentile(ent, 70):
    print("  → espectro DESORDENADO: domina el ruido (rango/lateral)")
else:
    print("  → espectro intermedio")
print(f"  Cambios de régimen espectral detectados: {len(cambios)}"
      + (f" | último hace ~{len(returns)-cambios[-1]} h" if cambios else ""))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(centros, ent, lw=1.4, color="#1565c0", label="entropía espectral (ruido↑)")
ax.plot(centros, low, lw=1.4, color="#2e7d32", label="peso ciclos >24h (tendencia↑)")
for c in cambios:
    ax.axvline(c, color="#c62828", ls=":", lw=1)
ax.set_title("Espectro rodante: cambios de régimen (líneas rojas)")
ax.set_xlabel("velas H1"); ax.legend(loc="best")
fig.tight_layout(); plt.show()

# --- (ii) régimen por CADENA DE MARKOV oculta (HMM de 3 estados)
def fit_hmm(returns, n_states, seed):
    """Devuelve (transición, mus, sigmas, estados) ordenados: 0=alcista, 1=bajista, 2=rango."""
    from hmmlearn.hmm import GaussianHMM
    SCALE = 100.0
    drift = pd.Series(returns).rolling(24, min_periods=1).mean().values
    X = np.column_stack([returns * SCALE, drift * SCALE * 20.0])
    best_model, best_ll = None, -np.inf
    for s in range(5):
        m = GaussianHMM(n_components=n_states, covariance_type="diag",
                        n_iter=500, random_state=seed + s, tol=1e-6, min_covar=1e-6)
        m.fit(X)
        ll = m.score(X)
        if ll > best_ll:
            best_model, best_ll = m, ll
    model  = best_model
    trans  = model.transmat_
    mus    = model.means_[:, 0] / SCALE
    covs   = np.array([np.diag(c)[0] for c in model.covars_])
    sigmas = np.sqrt(covs) / SCALE
    order_key = model.means_[:, 1]
    hidden = model.predict(X)
    print(f"\n[HMM] calibrado | log-verosimilitud: {best_ll:.1f}")
    order = [int(np.argmax(order_key)), int(np.argmin(order_key))]
    order.append([s for s in range(n_states) if s not in order][0])
    idx   = np.array(order)
    remap = {old: new for new, old in enumerate(idx)}
    return (trans[np.ix_(idx, idx)], mus[idx], sigmas[idx],
            np.array([remap[s] for s in hidden]))


trans, mus, sigmas, hidden = fit_hmm(returns, CONFIG["n_states"], CONFIG["seed"])
current_state = int(hidden[-1])
nombres = ["ALCISTA", "BAJISTA", "RANGO"]
print("===== REGÍMENES DE MARKOV CALIBRADOS =====")
for s in range(3):
    occup = 100.0 * np.mean(hidden == s)
    dur   = 1.0 / max(1e-9, 1.0 - trans[s, s])
    print(f"  {nombres[s]:8s}| mu={mus[s]*100:+.4f}%/h  sigma={sigmas[s]*100:.4f}%  "
          f"ocupación={occup:5.1f}%  duración media={dur:6.1f}h")
print(f"\n  >>> RÉGIMEN ACTUAL (Markov): {nombres[current_state]} <<<")


In [ ]:
# @title 8) MONTE CARLO sobre la cadena de Markov (+ drift de ciclos de Fourier)
def simulate(trans, mus, sigmas, p0, s0, n_paths, n_bars, ou_kappa, seed,
             cycle_drift=None):
    """Simula n_paths caminos H1. Cada camino salta de régimen según la matriz
       de transición de Markov; si cycle_drift viene de Fourier, se suma como
       componente determinista hora a hora."""
    rng    = np.random.default_rng(seed)
    states = np.full(n_paths, s0)
    logp   = np.full(n_paths, np.log(p0))
    anchor = logp.copy()
    cum_trans = np.cumsum(trans, axis=1)
    prices = np.empty((n_bars + 1, n_paths))
    prices[0] = p0
    for t_ in range(1, n_bars + 1):
        u = rng.random(n_paths)
        new_states = np.empty(n_paths, dtype=int)
        for s in range(3):
            mask = states == s
            if mask.any():
                new_states[mask] = np.searchsorted(cum_trans[s], u[mask])
        entrando_rango = (new_states == 2) & (states != 2)
        anchor[entrando_rango] = logp[entrando_rango]
        states = new_states
        z  = rng.standard_normal(n_paths)
        cd = cycle_drift[t_ - 1] if cycle_drift is not None else 0.0
        trend = states != 2
        logp = np.where(
            trend,
            logp + mus[np.clip(states, 0, 1)] + sigmas[np.clip(states, 0, 1)] * z + cd,
            logp + ou_kappa * (anchor - logp) + sigmas[2] * z + cd,
        )
        prices[t_] = np.exp(logp)
    return prices


p0 = float(closes[-1])
cd_txt = "con drift de ciclos Fourier" if cycle_drift is not None else "sin drift de ciclos"
print(f"[MC] Simulando {CONFIG['n_paths']} caminos x {CONFIG['horizon_hours']}h "
      f"desde {p0:.2f} en régimen {nombres[current_state]} ({cd_txt})...")
paths = simulate(trans, mus, sigmas, p0, current_state,
                 CONFIG["n_paths"], CONFIG["horizon_hours"],
                 CONFIG["ou_kappa"], CONFIG["seed"], cycle_drift)
print("[MC] Listo.")


In [ ]:
# @title 9) RUTA hora a hora, niveles alcanzables y objetivo
def report_route(paths, p0, cfg, target=None):
    conf   = cfg["confidence"]
    q_lo   = (1.0 - conf) / 2.0 * 100.0
    q_hi   = 100.0 - q_lo
    step   = cfg["checkpoint_every"]
    n_bars = paths.shape[0] - 1

    print(f"===== RUTA PRONOSTICADA ({paths.shape[1]} simulaciones, banda {conf*100:.0f}%) =====")
    print(f"  Inicio: XAUUSD {p0:.2f}\n")
    print(f"  {'hora':>6} | {'mediana':>9} | {'banda ' + format(conf*100,'.0f') + '%':^23} | {'P(subida)':>9}")
    print(f"  {'-'*6} | {'-'*9} | {'-'*23} | {'-'*9}")
    for h in range(step, n_bars + 1, step):
        med = np.median(paths[h])
        lo  = np.percentile(paths[h], q_lo)
        hi  = np.percentile(paths[h], q_hi)
        pup = 100.0 * np.mean(paths[h] > p0)
        print(f"  En {h:2d}h | {med:9.2f} | [{lo:9.2f} - {hi:9.2f}] | {pup:7.1f}%")

    run_max, run_min = paths.max(axis=0), paths.min(axis=0)
    lvl_up   = np.percentile(run_max, (1.0 - conf) * 100.0)
    lvl_down = np.percentile(run_min, conf * 100.0)
    print(f"\n===== NIVELES ALCANZABLES EN {n_bars}h =====")
    print(f"  Con {conf*100:.0f}% de prob. TOCA >= {lvl_up:.2f} ({(lvl_up-p0)/0.10:+.0f} pips)")
    print(f"  Con {conf*100:.0f}% de prob. TOCA <= {lvl_down:.2f} ({(lvl_down-p0)/0.10:+.0f} pips)")
    print(f"  Con 50% de prob. TOCA >= {np.percentile(run_max, 50):.2f} "
          f"| TOCA <= {np.percentile(run_min, 50):.2f}")

    if target is not None:
        if target > p0:
            p_touch = 100.0 * np.mean(run_max >= target)
            touch_h = [np.argmax(paths[:, j] >= target)
                       for j in range(paths.shape[1]) if run_max[j] >= target]
        else:
            p_touch = 100.0 * np.mean(run_min <= target)
            touch_h = [np.argmax(paths[:, j] <= target)
                       for j in range(paths.shape[1]) if run_min[j] <= target]
        print(f"\n===== OBJETIVO {target:.2f} =====")
        print(f"  Probabilidad de TOCARLO en {n_bars}h: {p_touch:.1f}%")
        if touch_h:
            print(f"  Hora típica del toque (mediana): {int(np.median(touch_h))}h "
                  f"| p10: {int(np.percentile(touch_h,10))}h "
                  f"| p90: {int(np.percentile(touch_h,90))}h")


report_route(paths, p0, CONFIG, TARGET)


In [ ]:
# @title 10) Gráficos y SÍNTESIS FINAL
n_bars = paths.shape[0] - 1
hours  = np.arange(n_bars + 1)

fig, ax = plt.subplots(figsize=(12, 5.5))
for lo_, hi_, a in [(2.5, 97.5, 0.15), (10, 90, 0.25), (25, 75, 0.35)]:
    ax.fill_between(hours, np.percentile(paths, lo_, axis=1),
                    np.percentile(paths, hi_, axis=1),
                    color="#1565c0", alpha=a, label=f"banda {hi_-lo_:.0f}%")
ax.plot(hours, np.median(paths, axis=1), "k-", lw=2, label="mediana (ruta central)")
ax.plot(np.arange(1, n_bars + 1), np.exp(proyeccion[:n_bars]), ls="--", lw=1.6,
        color="#ef6c00", label="proyección ciclos Fourier")
rng_ = np.random.default_rng(1)
for j in rng_.choice(paths.shape[1], size=25, replace=False):
    ax.plot(hours, paths[:, j], lw=0.4, alpha=0.4, color="#616161")
ax.axhline(p0, color="k", ls=":", lw=1)
if TARGET is not None:
    ax.axhline(TARGET, color="#c62828", ls="--", lw=1.5, label=f"objetivo {TARGET:.2f}")
ax.set_title(f"XAUUSD a {n_bars}h: Monte Carlo Markov + ciclos de Fourier")
ax.set_xlabel("horas"); ax.set_ylabel("USD/oz"); ax.legend(loc="upper left")
fig.tight_layout(); fig.savefig("xauusd_pipeline_fanchart.png", dpi=120); plt.show()

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.hist(paths[-1], bins=120, color="#2e7d32", alpha=0.85)
ax.axvline(p0, color="k", ls=":", lw=1.5, label=f"inicio {p0:.2f}")
ax.axvline(np.median(paths[-1]), color="#c62828", lw=1.5,
           label=f"mediana {np.median(paths[-1]):.2f}")
if TARGET is not None:
    ax.axvline(TARGET, color="#ef6c00", ls="--", lw=1.5, label=f"objetivo {TARGET:.2f}")
ax.set_title(f"Distribución del precio a {n_bars}h")
ax.set_xlabel("USD/oz"); ax.legend()
fig.tight_layout(); fig.savefig("xauusd_pipeline_final.png", dpi=120); plt.show()

med_fin = float(np.median(paths[-1]))
pup_fin = 100.0 * np.mean(paths[-1] > p0)
mc_dir  = "ARRIBA" if pup_fin > 55 else ("ABAJO" if pup_fin < 45 else "LATERAL")
acuerdo = "COINCIDEN" if mc_dir == veredicto else "DIVERGEN (más incertidumbre: reduce riesgo)"

print("=" * 62)
print("                    SÍNTESIS FINAL")
print("=" * 62)
print(f"  1. FOURIER (ciclos):      dirección {veredicto} "
      f"(señal/ruido {ratio:+.2f})")
print(f"  2. RÉGIMEN espectral:     entropía {ent[-1]:.3f} "
      f"({'ordenado' if ent[-1] < np.percentile(ent, 30) else 'ruidoso' if ent[-1] > np.percentile(ent, 70) else 'intermedio'})")
print(f"  3. RÉGIMEN Markov (HMM):  {nombres[current_state]}")
print(f"  4. MONTE CARLO a {n_bars}h:   mediana {med_fin:.2f} "
      f"({(med_fin/p0-1)*100:+.2f}%) | P(subida) {pup_fin:.1f}% → {mc_dir}")
print(f"  5. Fourier vs Monte Carlo: {acuerdo}")
print("=" * 62)
print("  Recuerda: distribución de probabilidad, no certeza. La banda 80%")
print("  falla 1 de cada 5 veces por diseño, y las noticias no se modelan.")
